#### Comenzamos importando las librerías necesaris.
En el caso de Keras, es necesario hacer alias de varios módulos de Python que se han movido desde la versión 3.9 a la versión 3.10. El módulo collections ha pasado a llamarse collections.abc. 

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

# Cargar los datos
data = pd.read_csv("vehicles.csv")


#### Cargar el dataset
Vamos a trabajar con used_cars.csv, pero necesita una fuerte limpieza. Lo podemos hacer aquí o en otra parte.

In [ ]:
df = pd.read_csv('vehicles.csv')
df.head()

#### Split the data into features and target variable:

In [ ]:
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

##### Split the data into training and testing sets:

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

#### Scale the data

In [ ]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

#### Define the model:

In [ ]:
model = Sequential()
model.add(Dense(units=32, kernel_initializer='uniform', activation='relu', input_dim=6))
model.add(Dense(units=32, kernel_initializer='uniform', activation='relu'))
model.add(Dense(units=1, kernel_initializer='uniform'))

#### Compile the model

In [ ]:
model.compile(optimizer='adam', loss='mean_squared_error')

#### Train the model


In [ ]:
history = model.fit(X_train, y_train, batch_size=32, epochs=100, validation_data=(X_test, y_test))

#### Evaluate the model

In [ ]:
loss = model.evaluate(X_test, y_test, verbose=0)
print(f'Test loss: {loss:.3f}')

#### Make predictions

In [ ]:
# make predictions
y_pred = model.predict(X_test)


##### Visualize the training and variations loss over time

In [ ]:
# Visualize the training and variations loss over time
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='test')
plt.legend()
plt.show()

#### Plot the results

In [ ]:
# plot the results
plt.figure(figsize=(10, 5))
plt.plot(y_test, label='Actual')
plt.plot(y_pred, label='Predicted')
plt.legend()
plt.show()
#### Save the model
model.save('car_price_model.h5')
#### Load the model
from keras.models import load_model
model = load_model('car_price_model.h5')
#### Make predictions
y_pred = model.predict(X_test)

#### Hyperparameter Tuning
Tuning hyperparameters can be a challenging and time-consuming task in machine learning. Genetic algorithms are one way to automate this process and search for optimal hyperparameters. Here are the general steps to use genetic algorithms for hyperparameter tuning:

   1. Define the search space: Specify the range of values that each hyperparameter can take. For example, the learning rate can vary between 0.001 and 0.1.
   2. Define the fitness function: Create a function that evaluates the performance of the model given a set of hyperparameters. This function should take the hyperparameters as input and return a fitness score that measures the quality of the model's performance.
   3. Generate an initial population: Create an initial set of random hyperparameters for the model.
   4. Select parents: Select a subset of the population to be parents for the next generation. This selection can be based on fitness score or other criteria.
   5. Apply genetic operators: Apply crossover and mutation to the selected parents to create a new set of hyperparameters.
   6. Evaluate fitness: Evaluate the fitness of the new set of hyperparameters.
   7. Select survivors: Select the best hyperparameters from the new population to be the parents for the next generation.
   8. Repeat: Repeat steps 5-7 until convergence or the desired number of generations is reached.

Here's an example of how to use genetic algorithms for hyperparameter tuning with the DEAP library in Python:

In [ ]:
import random
from deap import base, creator, tools

# Define the search space
param_min = [0.001, 5, 16]
param_max = [0.1, 50, 256]
param_types = [float, int, int]

# Define the fitness function


def evaluate(params):
    # Create and train the model with the given hyperparameters
    model = create_model(params)
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
    # Evaluate the model on the validation set
    score = model.evaluate(X_val, y_val, verbose=0)
    return score[0]


# Create the DEAP toolbox
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)
toolbox = base.Toolbox()

# Define the genetic operators
toolbox.register("attr_float", random.uniform, param_min[0], param_max[0])
toolbox.register("attr_int", random.randint, param_min[1], param_max[1])
toolbox.register("individual", tools.initCycle, creator.Individual,
                 [toolbox.attr_float if t == float else toolbox.attr_int for t in param_types], n=1)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=0.1, indpb=0.1)
toolbox.register("select", tools.selTournament, tournsize=3)

# Define the algorithm parameters
population_size = 50
cxpb = 0.5
mutpb = 0.2
num_generations = 10

# Generate an initial population
population = toolbox.population(n=population_size)

# Evaluate the fitness of the initial population
fitnesses = list(map(evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = (fit,)

# Run the genetic algorithm
for i in range(num_generations):
    # Select the next generation of parents
    parents = toolbox.select(population, len(population))
    # Apply crossover and mutation to the parents to generate offspring
    offspring = [toolbox.clone(ind) for ind in parents]
    for ind1, ind2 in zip(offspring[::2], offspring[1::2]):
        if random.random() < cxpb:
            toolbox.mate(ind1, ind2)
            del ind1.fitness.values, ind2.fitness.values
    for ind in offspring:
        if random.random() < mutpb:
            toolbox.mutate(ind)
            del ind.fitness.values
    # Evaluate the fitness of the new offspring
    fitnesses = map(evaluate, offspring)
    for ind, fit in zip(offspring, fitnesses):
        ind.fitness.values = (fit,)
    # Replace the current population with the offspring
    population[:] = offspring

# Print the best individual and its fitness score
best_ind = tools.selBest(population, k=1)[0]
best_score = best_ind.fitness.values[0]
print("Best individual:", best_ind)
print("Best score:", best_score)